# 허깅페이스 디자인 패턴

1 Auto...을 쓰면 키워드와 매핑된 클래스를 자동으로 불러와줌

# Pipeline

Pipeline = To

In [ ]:
dsadasd

# Tokenizer

### BERT Tokenizer

In [5]:
# 허깅페이스 라이브러리들 공통된 디자인 패턴
# from_pretraind 등 자주 쓰는 메소드
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

In [4]:
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [ ]:
from transfomers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

- https://huggingface.co/bert-base-cased을 가보면 https://huggingface.co/google-bert/bert-base-cased로 리다이렉트됨
- 여기에 가보면 tokenizer_config.json, tokenizer.json, special_tokens_map.json, added_tokens.json(BERT는 없음)
- `tokenizer_config.json` : 토크나이저 클래스, 특수 토큰 등 정의
- `special_tokens_map.json` : <unk>, <pad>, <eos> : 특수 토큰
- `added_tokens.json` : 기본 vocab 이후 사용자가 tokenizer.add_tokens() 추가한 토큰 목록만 따로 저장.
- `tokenizer.json` : fast 토크나이저 전용 JSON, vocab, merege table, 특수토큰, normalizer 설정이 모두 있음
- `tokenizer.model` : 바이너리 파일, (모델 파라미터 + vocab 포함)


vocab.json : BPE, Wordpiece 계열이면 vocab.json(토큰=>ID), merges.txt(merge 규칙) 상이 생김. SentencePiece에는 해당 안됨


In [11]:
from huggingface_hub import model_info, hf_hub_url

print(model_info("bert-base-cased").modelId)

google-bert/bert-base-cased


In [16]:
tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

실제 동작하면 Tokenization + Conversion 까지 모두 수행 : Encoding

![Image](https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdn%2FbPonBM%2FbtrGw3n6X5G%2FTmWKTcdq48tW2lhlCFFhQK%2Fimg.png)

In [6]:
tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

print(tokens)

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']


- Conversion의 경우는 vocab.txt를 활용함 
- 토크나이저 클래스의 생성자에서 vocab.txt에서 불러와서 vocab idx<-> token dict를 들고 있음
- 이걸로 매핑해서 정수 값으로 치환함


```python
def load_vocab(vocab_file):
    """Loads a vocabulary file into a dictionary."""
    vocab = collections.OrderedDict()
    with open(vocab_file, "r", encoding="utf-8") as reader:
        tokens = reader.readlines()
    for index, token in enumerate(tokens):
        token = token.rstrip("\n")
        vocab[token] = index
    return vocab

```

In [15]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


In [17]:
decoded_string = tokenizer.decode([7993, 170, 11303, 1200, 2443, 1110, 3014])
print(decoded_string)

Using a transformer network is simple


In [ ]:
# 필요한것들 저장
tokenizer.save_pretrained("/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/1_llm_intro/out")

('/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/1_llm_intro/out/tokenizer_config.json',
 '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/1_llm_intro/out/special_tokens_map.json',
 '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/1_llm_intro/out/vocab.txt',
 '/purestorage/AILAB/AI_1/tyk/3_CUProjects/language_model/LLM/1_llm_intro/out/added_tokens.json')

- https://github.dev/huggingface/transformers에 가서 해당 클래스에 대한 정의를 확인
- `huggingface/transformers/src/transformers/models/bert` 위치에 BERT 모델들이 모두 정의되어 있음
- `modeling_bert.py` : BERT Transformer 모델
- `tokenization_bert.py` : 해당 모델이 사용한 토크나이저

### 다른 토크나이저

```
AutoTokenizer.from_pretrained(tokenizer_file="tokenizer.json")
```


import sentencepiece as spm; spm.SentencePieceProcessor(model_file="tokenizer.model")

In [19]:
https://github.dev/huggingface/transformers

SyntaxError: invalid syntax (2966886778.py, line 1)

# Model

model load 

auto : 추론용, DDP 훈련에서 쓰면 grad sync가 끊겨 오류

In [ ]:
# pipeline을 쓰면 기본 모델을 로드해서 사용함

In [ ]:
from transformers import pipeline
qa = pipeline("question-answering")   # ← 기본 모델 자동 선택
print(type(qa.model))                # <class 'transformers.models.distilbert.modeling_distilbert.DistilBertForQuestionAnswering'>
print(qa.model.name_or_path)         # distilbert-base-cased-distilled-squad
print(type(qa.tokenizer))            # <class 'transformers.models.bert.tokenization_bert_fast.BertTokenizerFast'>


In [ ]:
# 세부적으로 지정할 수 있음

qa = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",   # 원하는 체크포인트
    tokenizer="deepset/roberta-base-squad2",
    revision="refs/convert/parquet-2025-03-22"  # 고정된 revision 예시
)


![Image](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline-dark.svg)

In [ ]:
from_pretraiend
device_map를 통해 매핑표를 전달

"auto", "balanced", …	: Accelerate가 알아서 분산(레이어 스플릿)
{"": "cuda:0"} : 루트 모듈 전체를 단일 GPU 0에 둬라
{"model.embed_tokens":0, "model.layers.0":0, "model.layers.1":1} : 서브모듈별 수동 배치

# 서브모듈별로 조절가능
device_map = {
    "vision_model":   "cuda:0",   # 이미지 타워
    "language_model": "cuda:1",   # 텍스트 타워
    "":               "cpu"       # 나머지(혹은 디스크)
}
